In [1]:
# Install packages from requirements.txt
!pip install -r requirements.txt

  Using cached langchain-0.2.16-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_community-0.2.17-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain-0.2.16-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_community-0.2.17-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
ERROR: Cannot install langchain-huggingface==0.0.3 and langchain-huggingface==0.3.1 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested langchain-huggingface==0.3.1
    The user requested langchain-huggingface==0.0.3

Additionally, some packages in these conflicts have no matching distributions available for your environment:
    langchain-huggingface

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict

ERROR: ResolutionImpossible: for help visit

In [2]:
# Auto-reload modules when they change
%load_ext autoreload
%autoreload 2

In [3]:
# Import all functions from utils
from utils import *

# Verify get_llm is imported
print("Available functions:", [name for name in dir() if not name.startswith('_')])

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Available functions: ['AutoModelForCausalLM', 'AutoTokenizer', 'BertForSequenceClassification', 'BertTokenizer', 'ChatGroq', 'ConnectionError', 'Elasticsearch', 'ElasticsearchStore', 'F', 'In', 'Out', 'SentenceTransformer', 'advanced_query_routing', 'advanced_query_transformation', 'chromadb', 'config', 'exit', 'faiss', 'fusion_retrieval', 'generate_answer', 'get_ipython', 'get_llm', 'json', 'load_config', 'np', 'open', 'os', 'pipeline', 'quit', 'rerank_documents', 'select_and_compress_context', 'sentence_model']


In [4]:
import faiss
import numpy as np
from elasticsearch import Elasticsearch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BertTokenizer, BertForSequenceClassification
from sentence_transformers import SentenceTransformer
import chromadb
import os
import json

In [2]:
# Config is already loaded in utils.py, access it here
# print(f"GROQ_API_KEY: {config.get('GROQ_API_KEY')[:20]}...")
# print(f"Config loaded successfully!")


In [6]:
from huggingface_hub import login

# Your Hugging Face API token (You can find it in your Hugging Face account settings)
hf_api_token = config.get('HUGGING_FACE_API_KEY')


# Log in to Hugging Face
login(token=hf_api_token)

In [7]:
# Load the pre-trained models and tokenizers for text generation, sentence embedding,
# and reranking.

# Load the SentenceTransformer model for encoding queries and documents
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')  # Small, fast model for embeddings

# Load the tokenizer for the reranking model
rerank_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load the BERT model for sequence classification (used for reranking)
rerank_model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

# Load summarization model
summarizer = pipeline("summarization")

print("All models loaded successfully!")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
Device set to use cpu


All models loaded successfully!


In [8]:
# Test the LLM function
response = get_llm().invoke("Hi")
print(response.content)


It's nice to meet you. Is there something I can help you with, or would you like to chat?


In [9]:
get_llm().invoke("Hi")

AIMessage(content="It's nice to meet you. Is there something I can help you with, or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 11, 'total_tokens': 34, 'completion_time': 0.023433712, 'prompt_time': 0.681520066, 'queue_time': 2.487430889, 'total_time': 0.704953778}, 'model_name': 'meta-llama/llama-4-maverick-17b-128e-instruct', 'system_fingerprint': 'fp_d2c1f7e199', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--6d9733d7-4216-46c7-b89f-c132caa23fd3-0', usage_metadata={'input_tokens': 11, 'output_tokens': 23, 'total_tokens': 34})

In [10]:
import chromadb

# Initialize ChromaDB client and create collection
client = chromadb.Client()

# Define the collection name
collection_name = "movies"

try:
    # Attempt to create the collection in ChromaDB
    collection = client.create_collection(name=collection_name)
    print(f"Collection '{collection_name}' created successfully.")

    # Define the documents to be inserted into the collection
    documents = [
        {"id": "1", "content": "The Shawshank Redemption is a great movie to watch on a rainy day."},
        {"id": "2", "content": "Forrest Gump is an uplifting film perfect for a rainy afternoon."}
    ]

    # Extract the IDs and content for insertion
    ids = [doc["id"] for doc in documents]
    contents = [doc["content"] for doc in documents]

    # Insert documents into the collection
    collection.add(ids=ids, documents=contents)
    print("Documents inserted successfully.")

except Exception as e:
    print(f"Collection '{collection_name}' already exists. No need to create it again.")
    # Optionally, you could fetch the existing collection here
    collection = client.get_collection(name=collection_name)

except Exception as e:
    print(f"An error occurred: {e}")


Collection 'movies' created successfully.
Documents inserted successfully.
Documents inserted successfully.


In [ ]:
def advanced_rag_pipeline(query, collection, documents, es, index_name='movies'):
    """
    The main pipeline function for the Advanced Retrieval-Augmented Generation (RAG) system.
    It processes the query, retrieves relevant documents, reranks them, selects and compresses
    the context, and finally generates an answer.

    Args:
        query (str): The user's input query.
        collection: ChromaDB collection
        documents: List of documents
        es: Elasticsearch client
        index_name (str): Elasticsearch index name

    Returns:
        str: The final generated answer.
    """
    # Transform and route query
    transformed_query = advanced_query_transformation(query)
    retrieval_method = advanced_query_routing(transformed_query)

    # Retrieve documents using fusion retrieval
    retrieved_documents = fusion_retrieval(transformed_query, collection, documents, es, index_name)

    # Rerank documents based on relevance
    ranked_documents = rerank_documents(query, retrieved_documents, rerank_tokenizer, rerank_model)

    # Select and compress context for answer generation
    context = select_and_compress_context(ranked_documents, summarizer)

    # Get LLM instance
    llm = get_llm()
    
    # Generate final answer based on the context
    final_answer = generate_answer(query, context, llm)
    return final_answer


In [ ]:
from elasticsearch import Elasticsearch

# Load Elasticsearch credentials from config
es_host = config.get('ELASTICSEARCH_HOST')
es_api_key = config.get('ELASTICSEARCH_PASSWORD')  # This is actually the API key

# Initialize Elasticsearch client with API key
es = Elasticsearch(
    es_host,
    api_key=es_api_key
)

# Test connection
es.ping()

In [ ]:
# Example query
query = "What are some good movies to watch on a rainy day?"

# Define index name
index_name = 'movies'

# Run the query through the Advanced RAG Pipeline
answer = advanced_rag_pipeline(query, collection, documents, es, index_name)

# Output the generated answer
print(answer)


In [1]:
# Setup Kaggle credentials
# Option 1: Upload kaggle.json manually, then run:
!mkdir -p ~/.kaggle
# If you uploaded kaggle.json to the workspace root:
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
# Download the Movies Dataset from Kaggle
!kaggle datasets download -d rounakbanik/the-movies-dataset

# Unzip the dataset
!unzip -o the-movies-dataset.zip

# List the files
!ls -lh *.csv

print("\n✅ Dataset downloaded and extracted!")

Dataset URL: https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset
License(s): CC0-1.0
 60%|████████████████████████                | 137M/228M [00:00<00:00, 1.43GB/s]
100%|████████████████████████████████████████| 228M/228M [00:00<00:00, 1.43GB/s]

100%|████████████████████████████████████████| 228M/228M [00:00<00:00, 1.43GB/s]
Archive:  the-movies-dataset.zip
  inflating: credits.csv             Archive:  the-movies-dataset.zip
  inflating: credits.csv             
  inflating: keywords.csv            
  inflating: keywords.csv            
  inflating: links.csv               
  inflating: links_small.csv         
  inflating: movies_metadata.csv     
  inflating: links.csv               
  inflating: links_small.csv         
  inflating: movies_metadata.csv     
  inflating: ratings.csv             
  inflating: ratings.csv             
  inflating: ratings_small.csv       
  inflating: ratings_small.csv       

-rw-r--r-- 1 codespace codespace 182M Sep 21  2019 credits.csv

In [6]:
import pandas as pd
df = pd.read_csv('movies_metadata.csv')
df.head()

/tmp/ipykernel_2631/1841053763.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('movies_metadata.csv')


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [9]:
# Download required NLTK data
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
print("✅ NLTK data downloaded!")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


✅ NLTK data downloaded!


In [10]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import NLTKTextSplitter
import chromadb

df = df.loc[:5000,['original_title', 'overview']]

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'langchain'

In [11]:
# Test query
query = "What are some good action movies with great visual effects?"

print(f"Query: {query}\n")
print("Running Advanced RAG Pipeline...\n")

# Run the pipeline
answer = advanced_rag_pipeline(query, collection, documents, es, index_name)

print("="*80)
print("ANSWER:")
print("="*80)
print(answer)

Query: What are some good action movies with great visual effects?

Running Advanced RAG Pipeline...



NameError: name 'advanced_rag_pipeline' is not defined

# Test the Advanced RAG Pipeline with Real Movie Data

In [ ]:
# Create Elasticsearch index and insert movie data
index_name = 'movies'

# Delete index if exists
if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)
    print(f"Deleted existing index '{index_name}'")

# Create new index
es.indices.create(
    index=index_name,
    mappings={
        "properties": {
            "content": {"type": "text"},
            "title": {"type": "text"}
        }
    }
)
print(f"Created index '{index_name}'")

# Insert movie chunks into Elasticsearch
print("Inserting data into Elasticsearch...")
for idx, row in chunked_df.iterrows():
    es.index(
        index=index_name,
        id=str(idx),
        document={
            "content": row['chunks'],
            "title": row['original_title']
        }
    )

print(f"✅ Successfully stored {len(chunked_df)} movie chunks in Elasticsearch!")

In [ ]:
# Setup Elasticsearch
from elasticsearch import Elasticsearch

# Load Elasticsearch credentials from config
es_host = config.get('ELASTICSEARCH_HOST')
es_api_key = config.get('ELASTICSEARCH_PASSWORD')

# Initialize Elasticsearch client
es = Elasticsearch(es_host, api_key=es_api_key)

# Test connection
if es.ping():
    print("✅ Connected to Elasticsearch")
else:
    print("❌ Elasticsearch connection failed")

## Store Movie Data in Elasticsearch

In [ ]:
# Initialize ChromaDB and store movie data
client = chromadb.Client()

# Delete collection if it exists
try:
    client.delete_collection(name="movies")
    print("Deleted existing collection")
except:
    pass

# Create new collection
collection = client.create_collection(name="movies")

# Insert data into ChromaDB
print("Inserting data into ChromaDB...")
for idx, row in chunked_df.iterrows():
    collection.add(
        ids=[str(idx)],
        embeddings=[row['embeddings']],
        metadatas=[{
            'original_title': row['original_title'],
            'chunk': row['chunks']
        }]
    )

# Store documents list for RAG pipeline
documents = chunked_df['chunks'].tolist()

print(f"✅ Successfully stored {len(chunked_df)} movie chunks in ChromaDB!")

In [ ]:
# Create embeddings for each chunk
def encode_chunk(chunk):
    if not isinstance(chunk, str) or chunk.strip() == "":
        return None
    return sentence_model.encode(chunk).tolist()

print("Creating embeddings... This may take a few minutes.")
chunked_df['embeddings'] = chunked_df['chunks'].apply(encode_chunk)

# Drop rows with None embeddings
chunked_df.dropna(subset=['embeddings'], inplace=True)

print(f"Created embeddings for {len(chunked_df)} chunks")

## Create Embeddings and Store in ChromaDB

In [ ]:
# Chunk the overview column
text_splitter = NLTKTextSplitter(chunk_size=1500)

def split_overview(overview):
    if pd.isna(overview):
        return []
    return text_splitter.split_text(str(overview))

movies_df['chunks'] = movies_df['overview'].apply(split_overview)

# Flatten the dataframe
chunked_df = movies_df.explode('chunks').reset_index(drop=True)

# Remove empty chunks
chunked_df = chunked_df[chunked_df['chunks'].str.strip() != '']

print(f"Created {len(chunked_df)} chunks from {len(movies_df)} movies")
chunked_df.head()

## Chunk Movie Overviews

In [ ]:
import pandas as pd
from langchain.text_splitter import NLTKTextSplitter
import nltk

# Download NLTK data if needed
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')

# Load the movies dataset
# Update the path to where your movies_metadata.csv is located
movies_df = pd.read_csv('/workspaces/Modern-AI-Agents/chr5/movies_metadata.csv')

# Select relevant columns and limit to first 5000 for faster processing
movies_df = movies_df.loc[:5000, ['original_title', 'overview']]

print(f"Loaded {len(movies_df)} movies")
movies_df.head()

# Load Movies Dataset

We'll use the Kaggle Movies Dataset for real movie recommendations

In [12]:
def advanced_rag_pipeline(query, collection, documents, es, index_name='movies'):
    """
    The main pipeline function for the Advanced Retrieval-Augmented Generation (RAG) system.
    It processes the query, retrieves relevant documents, reranks them, selects and compresses
    the context, and finally generates an answer.

    Args:
        query (str): The user's input query.
        collection: ChromaDB collection
        documents: List of documents
        es: Elasticsearch client
        index_name (str): Elasticsearch index name

    Returns:
        str: The final generated answer.
    """
    # Transform and route query
    transformed_query = advanced_query_transformation(query)
    retrieval_method = advanced_query_routing(transformed_query)

    # Retrieve documents using fusion retrieval
    retrieved_documents = fusion_retrieval(transformed_query, collection, documents, es, index_name)

    # Rerank documents based on relevance
    ranked_documents = rerank_documents(query, retrieved_documents, rerank_tokenizer, rerank_model)

    # Select and compress context for answer generation
    context = select_and_compress_context(ranked_documents, summarizer)

    # Get LLM instance
    llm = get_llm()
    
    # Generate final answer based on the context
    final_answer = generate_answer(query, context, llm)
    return final_answer


In [13]:
import chromadb

# Initialize ChromaDB client and create collection
client = chromadb.Client()

# Define the collection name
collection_name = "movies"

try:
    # Attempt to create the collection in ChromaDB
    collection = client.create_collection(name=collection_name)
    print(f"Collection '{collection_name}' created successfully.")

    # Define the documents to be inserted into the collection
    documents = [
        {"id": "1", "content": "The Shawshank Redemption is a great movie to watch on a rainy day."},
        {"id": "2", "content": "Forrest Gump is an uplifting film perfect for a rainy afternoon."}
    ]

    # Extract the IDs and content for insertion
    ids = [doc["id"] for doc in documents]
    contents = [doc["content"] for doc in documents]

    # Insert documents into the collection
    collection.add(ids=ids, documents=contents)
    print("Documents inserted successfully.")

except Exception as e:
    print(f"Collection '{collection_name}' already exists. No need to create it again.")
    # Optionally, you could fetch the existing collection here
    collection = client.get_collection(name=collection_name)

except Exception as e:
    print(f"An error occurred: {e}")
    

Collection 'movies' already exists. No need to create it again.


In [14]:
from elasticsearch import Elasticsearch

# Load Elasticsearch credentials from config
es_host = config.get('ELASTICSEARCH_HOST')
es_api_key = config.get('ELASTICSEARCH_PASSWORD')  # This is actually the API key

# Initialize Elasticsearch client with API key
es = Elasticsearch(
    es_host,
    api_key=es_api_key
)

# Test connection
es.ping()

True

In [66]:
# Define the index name
index_name = 'movies'

# Check if index exists, if not create it
if not es.indices.exists(index=index_name):
    # Create index with mapping (updated syntax - no 'body' parameter)
    es.indices.create(
        index=index_name,
        mappings={
            "properties": {
                "content": {"type": "text"}
            }
        }
    )
    print(f"Index '{index_name}' created.")
else:
    print(f"Index '{index_name}' already exists.")

Index 'movies' already exists.


In [15]:
# Example query
query = "What are some good movies to watch on a rainy day?"

# Define index name
index_name = 'movies'

# Run the query through the Advanced RAG Pipeline
answer = advanced_rag_pipeline(query, collection, documents, es, index_name)

# Output the generated answer
print(answer)


A rainy day is the perfect excuse to stay indoors and get lost in a great movie. Based on the context provided, it seems like you're looking for films that are not only entertaining but also uplifting and perhaps a bit inspirational. Let's dive into some movie suggestions that fit the bill for a cozy rainy day.

1. **The Shawshank Redemption (1994)** - As you've mentioned, this is a great movie, and for good reason. It's a powerful tale of hope, redemption, and the power of the human spirit. Its themes of perseverance and friendship make it a compelling watch on a day when you might be feeling a bit cooped up.

2. **Forrest Gump (1994)** - You've noted that Forrest Gump is an uplifting film, and it's hard to disagree. The movie's narrative spans decades, taking the viewer on a journey through significant historical events with a kind-hearted and simple man as the guide. It's a film that can leave you feeling hopeful and reflective.

3. **The Notebook (2004)** - A classic romance that t